# DarkPipe 0.9 - red independiente de relojes GPS

Este Colab inspecciona y verifica el resultado prospectivo ya congelado de la campana `DP-GPS-NETWORK-TRANSIENT-0.9-20260825`. **No reabre ni relanza el objetivo historico**. Descarga unicamente el repositorio compacto; los productos crudos JPL no se conservan en el runtime.

Jurisdiccion: candidata/no-candidata de transitorio coherente dentro del operador GPS, banco de velocidades y ventana UTC congelados. Materia oscura, hiperestados plasmicos, mecanismo gravitatorio y limites fisicos siguen `NOT_ESTIMABLE`.

In [ ]:
!git clone --depth 1 --branch v0.9.0 https://github.com/FacundoFirmenich/darkpipe-realdata.git /content/darkpipe-realdata
%cd /content/darkpipe-realdata
!python -m pip install -q -e .

In [ ]:
import hashlib, json
from pathlib import Path

result_path = Path('evidence/gps_network_v09/target_result.json')
result = json.loads(result_path.read_text(encoding='utf-8'))
assert result['schema'] == 'darkpipe.gps_network.target.v1'
assert result['stage'] == 'TARGET_OPENED_ONCE_AFTER_GREEN_POWER_GATE'
assert result['decision'] == 'NO_GPS_NETWORK_TRANSIENT_CANDIDATE'
assert result['null_count'] == 42
assert abs(result['exact_familywise_rank_p'] - 13/43) < 1e-15
print('SHA-256 receipt:', hashlib.sha256(result_path.read_bytes()).hexdigest())
print('Decision:', result['decision'])
print('p exacto familiar:', result['exact_familywise_rank_p'])
print('Maximo:', result['winning_hit']['statistic'], 'en', result['winning_hit']['center_utc'])

In [ ]:
cal = json.loads(Path('evidence/gps_network_v09/calibration_result.json').read_text(encoding='utf-8'))
p8 = next(row for row in cal['power'] if row['amplitude_robust_sigma'] == 8.0)
p4 = next(row for row in cal['power'] if row['amplitude_robust_sigma'] == 4.0)
print('Gate 8 sigma:', p8)
print('Sensibilidad adversa 4 sigma:', p4)
assert p8['joint_wilson_95_lower'] >= 0.80
assert p4['joint_successes'] == 38

## Interpretacion sustantiva

El maximo objetivo no fue raro: doce de los 42 maximos nulos fueron al menos tan grandes, por lo que el rango corregido es `(1+12)/(1+42)=13/43`, aproximadamente 0.3023. El nulo es claro para este detector y esta ventana, pero la baja potencia a 4 sigma impide una exclusion fisica fuerte. La siguiente decision cientifica requiere un operador directo que traduzca la conjetura fisica a formas de onda y acoplamientos instrumentales, no un nuevo barrido post hoc de esta ventana.

In [ ]:
import zipfile
out = Path('/content/darkpipe_v09_compact_result.zip')
with zipfile.ZipFile(out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in [
        'evidence/gps_network_v09/calibration_result.json',
        'evidence/gps_network_v09/calibration_state.json',
        'evidence/gps_network_v09/target_result.json',
        'docs/PREREGISTRATION_GPS_NETWORK_TRANSIENT_0.9.md',
        'docs/V09_SUBSTANTIVE_CLOSURE_ES_2026-08-26.md',
        'LICENSE',
    ]:
        zf.write(name)
print(out, out.stat().st_size, 'bytes')
from google.colab import files
files.download(str(out))